![Astrofisica Computacional](../../new_logo.png)

# Computational Astrophysics –  ML Logistic Regression II

## Dr. rer. nat. Jose Ivan Campos Rozo<sup>1,2</sup>

1. Astronomical Institute of the Czech Academy of Sciences\
   Department of Solar Physics\
   Ondřejov, Czec Republic

2. Observatorio Astronómico Nacional\
   Facultad de Ciencias\
   Universidad Nacional de Colombia

e-mail: jicamposr@unal.edu.co & rozo@asu.cas.cz)

---
Taken from previous lectures of this course.



### About this notebook

In this worksheet, we will use the Logistic regression algorithm implemented before to classify a dataset with photometric information of some objects from the SDSS.

---

---
## Clasificación de objetos astronómicos

Como ejemplo de **regresión logística**, consideraremos un conjunto de datos de objetos astronómicos de la base de datos SDSS (DR17). Este conjunto incluye 100 000 objetos con 18 características. Se obtuvo de [esta página].(https://www.kaggle.com/datasets/fedesoriano/stellar-classification-dataset-sdss17).

In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

original_data = pd.read_csv("object_classification.csv")
original_data

,obj_ID,alpha,delta,u,g,r,i,z,run_ID,rerun_ID,cam_col,field_ID,spec_obj_ID,class,redshift,plate,MJD,fiber_ID
0,1.237661e+18,135.689107,32.494632,23.87882,22.27530,20.39501,19.16573,18.79371,3606,301,2,79,6.543777e+18,GALAXY,0.634794,5812,56354,171
1,1.237665e+18,144.826101,31.274185,24.77759,22.83188,22.58444,21.16812,21.61427,4518,301,5,119,1.176014e+19,GALAXY,0.779136,10445,58158,427
2,1.237661e+18,142.188790,35.582444,25.26307,22.66389,20.60976,19.34857,18.94827,3606,301,2,120,5.152200e+18,GALAXY,0.644195,4576,55592,299
3,1.237663e+18,338.741038,-0.402828,22.13682,23.77656,21.61162,20.50454,19.25010,4192,301,3,214,1.030107e+19,GALAXY,0.932346,9149,58039,775
4,1.237680e+18,345.282593,21.183866,19.43718,17.58028,16.49747,15.97711,15.54461,8102,301,3,137,6.891865e+18,GALAXY,0.116123,6121,56187,842
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,1.237679e+18,39.620709,-2.594074,22.16759,22.97586,21.90404,21.30548,20.73569,7778,301,2,581,1.055431e+19,GALAXY,0.000000,9374,57749,438
99996,1.237679e+18,29.493819,19.798874,22.69118,22.38628,20.45003,19.75759,19.41526,7917,301,1,289,8.586351e+18,GALAXY,0.404895,7626,56934,866
99997,1.237668e+18,224.587407,15.700707,21.16916,19.26997,18.20428,17.69034,17.35221,5314,301,4,308,3.112008e+18,GALAXY,0.143366,2764,54535,74
99998,1.237661e+18,212.268621,46.660365,25.35039,21.63757,19.91386,19.07254,18.62482,3650,301,4,131,7.601080e+18,GALAXY,0.455040,6751,56368,470


### Los datos

La columna `'class'` nos dice el tipo de objeto: 'STAR', 'GALAXY' o 'QSO'. Para este ejemplo, consideraremos solo los tipos 'STAR' y 'GALAXY' y, por lo tanto, definiremos un nuevo dataframe que incluya muestras de estos tipos y las siguientes características: 'u', 'g', 'r', 'i', 'z', 'redshift', 'class'.

### Qué hacer?

df -> original_data select only\['column5', 'column6',...\]

df -> donde df\['class'\] != 'QSO'

Tenga en cuenta que el nuevo dataframe tiene solo xxxx muestras.

La idea de este ejercicio es entrenar un algoritmo de regresión logística que dé la "clase" de un objeto (clasifica la muestra) usando las características "u", "g", "r", "i", "z", "redshift". Primero, verifiquemos el comportamiento de algunas variables gráficamente,

In [ ]:
plt.figure()
plt.scatter(df['u'], df['i'])
plt.show()

Tenga en cuenta que en muchos de los gráficos es posible identificar un valor atípico. Por lo tanto, debemos eliminarlo. Primero, identificamos la muestra,

# Remover outlier

df -> buscar y remover el/los outliers

### Algunos gráficos

Crearemos un gráfico con algunas de las características, mostrando la "clase" de cada muestra con un código de color. A continuación, introduciremos una nueva columna que defina el color para "ESTRELLA" y "GALAXIA".

### clase color

df\['classColor'\] entonces df 'class' == 'STAR' -> df 'classColor' == 'color1' y df 'class' == 'GALAXY' -> df 'classColor' == 'color2' 


In [ ]:
fig, ax = plt.subplots(1,4, figsize=(10,3))
ax[0].scatter(df['g'], df['u'], c=df['classColor'], alpha=0.5)
ax[0].set_ylabel(r'u')
ax[0].set_xlabel(r'g')
ax[1].scatter(df['r'], df['u'], c=df['classColor'], alpha=0.5)
ax[1].set_xlabel(r'r')
ax[2].scatter(df['i'], df['u'], c=df['classColor'], alpha=0.5)
ax[2].set_xlabel(r'i')
ax[3].scatter(df['z'], df['u'], c=df['classColor'], alpha=0.5)
ax[3].set_xlabel(r'z')
plt.show()

Está claro que no es tarea fácil clasificar las muestras según la característica de "class" utilizando la información de la banda/filtro. Sin embargo, intentaremos entrenar un algoritmo de **Regresión Logística** utilizando esta información. El primer paso es definir las variables dependientes e independientes. Primero, convertimos/creamos la variable class en enteros 0,1.

### clase color

df\['classInt'\] entonces df 'class' == 'STAR' -> df 'classInt' == 0 y df 'class' == 'GALAXY' -> df 'classInt' == 1 


In [ ]:
Xdf = np.asarray(df[['u','g','r','i','z']])
ydf = np.asarray(df[['classInt']])

Now, we will split these sets into train and test subsets,

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(Xdf, ydf, random_state=1803, test_size=0.3)

print("Shape of X_train : ", X_train.shape)
print("Shape of Y_train : ", y_train.shape)
print("Shape of X_test : ", X_test.shape)
print("Shape of Y_test : ", y_test.shape)

### Logistic Regression Algorithm


We will use the same logistic algorithm that we implement in the previous lesson,

In [ ]:
class LogisticRegression():
    '''
    Logistic regression class
    '''
    def __init__(self):
        pass
    
    def sigmoid(self, Z):
        return 1/(1 + np.exp(-Z))

    def Z(self, X):
        '''
        Function to fit
        '''
        return self.b + np.dot(X,self.W)
    
    def predict(self, X):
        return self.sigmoid(self.Z(X))
    
    def cost(self, X, y):
        '''
        Cost function
        '''
        Yp = self.predict(X)
        return -(1/self.n)*np.sum(y*np.log(Yp) + (1-y)*np.log(1-Yp))
    
    def grad_cost(self, X,y):
        '''
        Gradient of the cost function
        '''
        Yp = self.predict(X)
        grad_dW = (1/self.n)*np.dot(X.T, Yp-y)
        grad_db = (1/self.n)*np.sum(Yp-y)
        return grad_dW, grad_db
    
    def fit(self, X, y):
        '''
        Optimization function
        '''
        alpha= 0.009  # Learning rate
        tol = 1e-10    # Tolerance
        np.random.seed(413)
        self.m = X.shape[1] # Number of features
        self.n = X.shape[0] # Number od samples
        
        self.W = np.zeros([self.m,1])#np.random.rand(self.m)
        self.b = 0#np.random.rand(1)
        Y = self.sigmoid(self.Z(X))

        self.history = []
        self.history.append(self.cost(X, y))
        print('Initial cost = ', self.history[0])
        
        epoch = 0 # Epochs
        epsilon = 1
        while epsilon>tol and epoch<70000:
            # Gradient
            grad_dW, grad_db = self.grad_cost(X,y)

            self.W = self.W - alpha*grad_dW
            self.b = self.b - alpha*grad_db
            
            self.history.append(self.cost(X, y))
            epsilon = abs(self.history[epoch] - self.history[epoch+1])
            epoch +=1
        
        print('Final cost = ', self.history[-1])
        print('Number of epochs = ',epoch)
    
    def accuracy(self, X, y):
        Yp = self.predict(X)
        Yp = Yp > 0.5
        Yp = np.array(Yp, dtype = 'int64')
        acc = (1 - np.sum(abs(Yp - y))/len(y))*100
        print("Accuracy of the model is : ", round(acc, 2), "%")
        
    

### Ejercicio:

1. Cargar nuestra clase LogisticRegression()
2. Hacer el ajust con los datos de entrenamiento
3. Imprimir la precisión (accuracy) tanto de entrenamiento como de prueba
4. Graficar la historia de la evolución de la función costo (history)
5. Calcular y graficar la matriz de confusión.
6. Repetir todo el proceso pero removiendo la clase 'STAR' en lugar de la clase QSO
7. Repetir y comparar usando `from sklearn.linear_model import LogisticRegression`